<a href="https://colab.research.google.com/github/tiendinh-hcmut/ML-Food-Ingredient-Classification/blob/main/food_ingredient.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

EDA

In [ ]:
# 1. Install required libraries
!pip install datasets pandas matplotlib seaborn missingno -q

from datasets import load_dataset
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import missingno as msno
import io
from PIL import Image

In [ ]:
# 2. Load the dataset and check its basic structure
print("Loading the dataset from HuggingFace...")
dataset = load_dataset("Scuccorese/food-ingredients-dataset")

# Convert the 'train' split into a Pandas DataFrame for easier analysis
df = dataset['train'].to_pandas()

In [ ]:

# Display the first 5 rows to see what the data looks like
print("\n--- First 5 rows of the dataset ---")
display(df.head())

# Check the dimensions (number of rows and columns)
print("\n--- Data Structure ---")
print(f"Number of rows (Total images): {df.shape[0]}")
print(f"Number of columns (Features): {df.shape[1]}")

# Display column names and their data types
print("\n--- Columns and Data Types ---")
df.info()

In [ ]:
# 3. Count unique values for hierarchical levels (excluding the image column)
print("--- Number of Unique Classes ---")
columns_to_count = ['category', 'subcategory', 'ingredient']
print(df[columns_to_count].nunique())

In [ ]:
# 4 & 5. Check and visualize Missing Values
print("--- Missing Values Count ---")
missing_counts = df.isnull().sum()
print(missing_counts)

print("\n--- Missing Values Percentage (%) ---")
print((df.isnull().mean() * 100).round(2))

# Visualize
plt.figure(figsize=(10, 5))
msno.matrix(df, figsize=(10, 5), sparkline=False, fontsize=12, color=(0.2, 0.4, 0.6))
plt.title("Missing Values Distribution in Dataset", fontsize=16)
plt.show()

In [ ]:
# EDA part 2
# Category Distribution (countplot)
plt.figure(figsize=(10,5))
sns.countplot(x='category', data=df)

plt.xticks(rotation=45)
plt.title("Distribution of Categories")
plt.show()

In [ ]:
# Subcategory Distribution
plt.figure(figsize=(12,6))

top_sub = df['subcategory'].value_counts().head(15)

sns.barplot(x=top_sub.values, y=top_sub.index)

plt.title("Top Subcategories")
plt.show()

In [ ]:
# Ingredient Distribution
plt.figure(figsize=(12,6))

top_ing = df['ingredient'].value_counts().head(20)

sns.barplot(x=top_ing.values, y=top_ing.index)

plt.title("Top 20 Ingredients")
plt.show()

In [ ]:
# Histogram
df['category'].value_counts().plot(kind='bar')
plt.title("Category Frequency")
plt.show()

In [ ]:
# Outliers
ingredient_counts = df['ingredient'].value_counts()

plt.figure(figsize=(10,5))
sns.boxplot(x=ingredient_counts)

plt.title("Distribution of Ingredient Sample Counts")
plt.show()

In [ ]:
sample = df.sample(9)

plt.figure(figsize=(8,8))

for i, (_, row) in enumerate(sample.iterrows()):
    plt.subplot(3,3,i+1)

    # Extract image bytes from the dictionary and open as PIL Image
    image_bytes = row["image"]["bytes"]
    img = Image.open(io.BytesIO(image_bytes))

    plt.imshow(img)
    plt.title(row["ingredient"])
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Function to get image width and height from the dictionary column
def get_image_dimensions(image_dict):
    if image_dict and 'bytes' in image_dict:
        img = Image.open(io.BytesIO(image_dict['bytes']))
        return img.width, img.height
    return None, None

df['width'], df['height'] = zip(*df['image'].apply(get_image_dimensions))

print(df[['width','height']].describe())

In [ ]:
plt.figure(figsize=(10,4))

plt.subplot(1,2,1)
sns.histplot(df["width"], bins=30)
plt.title("Image Width Distribution")

plt.subplot(1,2,2)
sns.histplot(df["height"], bins=30)
plt.title("Image Height Distribution")

plt.show()

DATA PREPROCESSING

In [ ]:
pip install datasets pandas scikit-learn pillow tqdm matplotlib seaborn requests

In [ ]:
import os
import matplotlib.pyplot as plt
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from PIL import Image
from tqdm import tqdm
from io import BytesIO
import hashlib
from concurrent.futures import ThreadPoolExecutor

# 1. Download dataset from Hugging Face
print("Downloading dataset...")
dataset = load_dataset("Scuccorese/food-ingredients-dataset", split='train')

# Convert to Pandas DataFrame for easier text/metadata manipulation
df = dataset.to_pandas()
print(f"Initial number of samples: {len(df)}")
df.head()

In [ ]:
LABEL_COL = 'ingredient'
IMAGE_COL = 'image'

# 1. Standardize data format (Convert labels to lowercase, trim extra whitespace)
if df[LABEL_COL].dtype == 'object':
    df[LABEL_COL] = df[LABEL_COL].astype(str).str.lower().str.strip()
    # Replace spaces and slashes with underscores for safe directory naming
    df[LABEL_COL] = df[LABEL_COL].str.replace(' ', '_').str.replace('/', '_')

# --- UNHASHABLE DICT HANDLING FUNCTION ---
# Create a unique hash for each image based on its byte data
def get_image_hash(img_data):
    try:
        # Case: Data is a dict containing bytes (Standard for Hugging Face datasets)
        if isinstance(img_data, dict) and 'bytes' in img_data and img_data['bytes']:
            return hashlib.md5(img_data['bytes']).hexdigest()
        # Case: Data has already been loaded as a PIL Image object
        elif hasattr(img_data, 'tobytes'):
            return hashlib.md5(img_data.tobytes()).hexdigest()
        # Fallback to string representation
        return str(img_data)
    except Exception:
        return str(img_data)

# 2. Remove Duplicate rows
print("Checking for duplicate data (this may take a few seconds)...")

# Create a temporary column containing the image hash string
df['img_hash'] = df[IMAGE_COL].apply(get_image_hash)

initial_len = len(df)

# Drop duplicates based on label and image hash (both are strings, which Pandas can handle)
df = df.drop_duplicates(subset=[LABEL_COL, 'img_hash']).reset_index(drop=True)

# Remove the temporary hash column
df = df.drop(columns=['img_hash'])

print(f"Removed {initial_len - len(df)} duplicate samples.")

In [ ]:
# Check for missing values
print("Missing data statistics:")
print(df.isnull().sum())

# Solution: Drop rows with missing Labels (Ingredients) or Images
# Reason: In Computer Vision tasks, missing images cannot be "imputed" using standard
# statistical methods. Furthermore, missing labels cannot be used for training
# supervised learning models.
df = df.dropna(subset=[LABEL_COL, IMAGE_COL]).reset_index(drop=True)
print(f"Number of samples after handling missing values: {len(df)}")

# Filter out classes with insufficient data (e.g., < 10 images)
# This prevents errors during stratified train/test splitting and ensures
# the model has enough samples to learn each class.
class_counts = df[LABEL_COL].value_counts()
valid_classes = class_counts[class_counts >= 10].index
df = df[df[LABEL_COL].isin(valid_classes)].reset_index(drop=True)

print(f"Number of samples after filtering minority classes: {len(df)}")

In [ ]:
# Split data into ratios: 70% Train, 15% Validate, 15% Test
# Use stratify=df[LABEL_COL] to ensure equal class distribution across all three sets

# Step 1: Separate Train (70%) and Temp (30%)
df_train, df_temp = train_test_split(df, test_size=0.3, random_state=42, stratify=df[LABEL_COL])

# Step 2: Separate Temp into Validate (15%) and Test (15%)
# (Since 50% of the 30% remaining is 15%)
df_val, df_test = train_test_split(df_temp, test_size=0.5, random_state=42, stratify=df_temp[LABEL_COL])

print(f"Train set: {len(df_train)} samples")
print(f"Validation set: {len(df_val)} samples")
print(f"Test set: {len(df_test)} samples")

In [ ]:
# Adjust based on CPU threads
MAX_WORKERS = 16

# Helper function to download/save and normalize images
def process_and_save_image(img_data, save_path):
    try:
        if isinstance(img_data, dict) and 'bytes' in img_data:
            # Case: Raw bytes from Hugging Face
            img = Image.open(BytesIO(img_data['bytes']))
        else:
            # Case: Already a PIL Image object
            img = img_data

        # Image Normalization: Convert to RGB, Resize to 224x224 (Standard for CNN/ViT)
        img = img.convert('RGB')
        img = img.resize((224, 224))
        img.save(save_path, format='JPEG')
        return True
    except Exception as e:
        return False

# Helper function to wrap row processing for multi-threading
def process_row(args):
    index, row, split_name, base_dir = args
    label = row[LABEL_COL]
    img_data = row[IMAGE_COL]

    # Create class folder: e.g., ./processed_dataset/train/tomato/
    class_dir = os.path.join(base_dir, split_name, str(label))
    os.makedirs(class_dir, exist_ok=True)

    # Save file
    file_path = os.path.join(class_dir, f"img_{index}.jpg")
    return process_and_save_image(img_data, file_path)

# Function to export dataset using Multi-threading
def export_dataset_parallel(dataframe, split_name, base_dir="./processed_dataset", max_workers=8):
    print(f"Parallel processing set: {split_name}...")

    # Prepare arguments list for each thread
    tasks = [
        (index, row, split_name, base_dir)
        for index, row in dataframe.iterrows()
    ]

    # Use ThreadPoolExecutor to run tasks in parallel
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Use list(tqdm(...)) to display a progress bar during parallel execution
        list(tqdm(executor.map(process_row, tasks), total=len(tasks)))

# Run multi-threaded data export
export_dataset_parallel(df_train, 'train', max_workers=MAX_WORKERS)
export_dataset_parallel(df_val, 'val', max_workers=MAX_WORKERS)
export_dataset_parallel(df_test, 'test', max_workers=MAX_WORKERS)

print("Parallel folder creation and image saving complete!")

In [ ]:
plt.figure(figsize=(16, 6))

# Chart 1: Distribution of Train/Val/Test data
plt.subplot(1, 2, 1)
splits = ['Train', 'Validate', 'Test']
counts = [len(df_train), len(df_val), len(df_test)]
plt.pie(counts, labels=splits, autopct='%1.1f%%', colors=['#4CAF50', '#FFC107', '#F44336'])
plt.title("Dataset Split Ratio")

plt.tight_layout()
plt.show()

CNN Feature Extractor

In [ ]:
!pip install tensorflow -q

import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import ResNet50, EfficientNetB0
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess
from tensorflow.keras.applications.efficientnet import preprocess_input as effnet_preprocess
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import os

# 1. Define paths and parameters
train_dir = './processed_dataset/train'
test_dir = './processed_dataset/test'
img_size = (224, 224)
batch_size = 32

print("Initializing Data Generators...")
# ResNet50 Data Generators
datagen_resnet = ImageDataGenerator(preprocessing_function=resnet_preprocess)
train_gen_resnet = datagen_resnet.flow_from_directory(
    train_dir, target_size=img_size, batch_size=batch_size, class_mode=None, shuffle=False)
test_gen_resnet = datagen_resnet.flow_from_directory(
    test_dir, target_size=img_size, batch_size=batch_size, class_mode=None, shuffle=False)

# EfficientNetB0 Data Generators
datagen_effnet = ImageDataGenerator(preprocessing_function=effnet_preprocess)
train_gen_effnet = datagen_effnet.flow_from_directory(
    train_dir, target_size=img_size, batch_size=batch_size, class_mode=None, shuffle=False)
test_gen_effnet = datagen_effnet.flow_from_directory(
    test_dir, target_size=img_size, batch_size=batch_size, class_mode=None, shuffle=False)

# 2. Initialize models
print("\nLoading ResNet50 and EfficientNetB0 models...")
resnet_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3), pooling='avg')
effnet_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224, 224, 3), pooling='avg')

# 3. Extract features (Predict)
print("\nStarting feature extraction with ResNet50...")
resnet_X_train = resnet_model.predict(train_gen_resnet, verbose=1)
resnet_X_test = resnet_model.predict(test_gen_resnet, verbose=1)

print("\nStarting feature extraction with EfficientNetB0...")
effnet_X_train = effnet_model.predict(train_gen_effnet, verbose=1)
effnet_X_test = effnet_model.predict(test_gen_effnet, verbose=1)

# 4. Save results to .npy files
print("\nSaving .npy files...")
np.save('resnet_X_train.npy', resnet_X_train)
np.save('resnet_X_test.npy', resnet_X_test)

np.save('effnet_X_train.npy', effnet_X_train)
np.save('effnet_X_test.npy', effnet_X_test)

print("\nComplete! The files have been saved successfully.")

TRANSFORMER FEATURE EXTRACTOR & LABEL MASTER

In [ ]:
# 1. Install necessary libraries
!pip install transformers torch -q

import tensorflow as tf
import numpy as np
import json
import torch
from transformers import ViTModel

# Check for GPU (PyTorch style)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type == 'cuda':
    print("🚀 Running on GPU (PyTorch), processing will be blazing fast!")
else:
    print("⚠️ Running on CPU.")

# 2. Define paths for all splits
train_dir = './processed_dataset/train'
val_dir = './processed_dataset/val'
test_dir = './processed_dataset/test'

# 3. Load datasets with shuffle=False (TensorFlow data loader)
print("\nLoading Train dataset...")
train_dataset = tf.keras.utils.image_dataset_from_directory(
    train_dir, image_size=(224, 224), batch_size=32, shuffle=False
)

print("Loading Validation dataset...")
val_dataset = tf.keras.utils.image_dataset_from_directory(
    val_dir, image_size=(224, 224), batch_size=32, shuffle=False
)

print("Loading Test dataset...")
test_dataset = tf.keras.utils.image_dataset_from_directory(
    test_dir, image_size=(224, 224), batch_size=32, shuffle=False
)

# 4. Create and save Label Map
print("\nCreating and saving label_map.json...")
class_names = train_dataset.class_names
label_map = {str(i): name for i, name in enumerate(class_names)}
with open('label_map.json', 'w') as f:
    json.dump(label_map, f, indent=4)

# 5. Extract y_train, y_val, and y_test
print("Saving y_train.npy, y_val.npy, and y_test.npy...")
y_train = np.concatenate([y for x, y in train_dataset], axis=0)
y_val = np.concatenate([y for x, y in val_dataset], axis=0)
y_test = np.concatenate([y for x, y in test_dataset], axis=0)

np.save('y_train.npy', y_train)
np.save('y_val.npy', y_val)
np.save('y_test.npy', y_test)

# 6. Initialize ViT (PyTorch Version)
print("\nLoading pretrained ViT model (PyTorch version)...")
# Load PyTorch ViT model and move to GPU
model = ViTModel.from_pretrained('google/vit-base-patch16-224-in21k').to(device)
model.eval() # Set model to evaluation mode

def extract_features(dataset):
    features = []
    for images, labels in dataset:
        # Convert images from TensorFlow Tensor to Numpy Array and normalize to [-1, 1]
        images_np = (images.numpy() / 127.5) - 1.0

        # PyTorch requires [batch, channels, height, width] format
        # (different from TF's [batch, height, width, channels]), so we transpose
        images_np = np.transpose(images_np, (0, 3, 1, 2))

        # Convert to PyTorch Tensor and move to GPU
        inputs = torch.tensor(images_np, dtype=torch.float32).to(device)

        with torch.no_grad(): # Disable gradient calculation to save RAM and speed up
            outputs = model(inputs)
            # Extract the [CLS] token
            cls_features = outputs.last_hidden_state[:, 0, :]
            # Move back to CPU and convert to Numpy for saving
            features.append(cls_features.cpu().numpy())

    return np.concatenate(features, axis=0)

print("\nExtracting features for Train set (X_train)...")
vit_X_train = extract_features(train_dataset)
np.save('vit_X_train.npy', vit_X_train)

print("Extracting features for Validation set (X_val)...")
vit_X_val = extract_features(val_dataset)
np.save('vit_X_val.npy', vit_X_val)

print("Extracting features for Test set (X_test)...")
vit_X_test = extract_features(test_dataset)
np.save('vit_X_test.npy', vit_X_test)

print("\n✅ DONE! All files generated successfully with PyTorch stability.")